# GOLD - ECOMMERCE CLIENTES

## Squad 1 - Data Quality em Tempo Real

### Objetivo
Construir a camada Gold com métricas agregadas de qualidade de dados
para consumo em painel de BI.

### Fontes
- Silver: ecommerce_clientes
- dq_monitoring_logs

### Saídas Gold
1. gold_dq_resumo_por_regra
   - Percentual de falhas por regra por dia.

2. gold_dq_resumo_por_tabela
   - Percentual de registros limpos por tabela por hora.

### Processamento
Silver + dq_monitoring_logs
→ agregação das métricas de qualidade
→ Gold Delta
→ consumo pelo BI

In [0]:
# ============================================================
# IMPORTS E CREDENCIAIS - GOLD
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from dotenv import load_dotenv
import os

# Carrega as variáveis do arquivo .env
load_dotenv(".env")

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")

# Validação sem exibir os valores das credenciais
print("client_id carregado:", client_id is not None)
print("tenant_id carregado:", tenant_id is not None)
print("client_secret carregado:", client_secret is not None)
print("storage_account carregado:", storage_account is not None)

In [0]:
# ============================================================
# CONFIGURAÇÃO ADLS - GOLD
# ============================================================

adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",

    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net":
        client_id,

    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net":
        client_secret,

    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

print("✅ Configuração do ADLS preparada com sucesso.")

In [0]:
# ============================================================
# CAMINHOS - GOLD
# ============================================================

# Tabela compartilhada de logs de qualidade
dq_logs_path = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/"
    "dq_monitoring_logs"
)

# Destinos das tabelas Gold
gold_regra_path = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/"
    "gold/dq_resumo_por_regra"
)

gold_tabela_path = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/"
    "gold/dq_resumo_por_tabela"
)

print("✅ Caminhos da Gold configurados")
print("DQ Logs:", dq_logs_path)
print("Gold por regra:", gold_regra_path)
print("Gold por tabela:", gold_tabela_path)

In [0]:
# ============================================================
# LEITURA DA TABELA DQ_MONITORING_LOGS
# ============================================================

df_dq_logs = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(dq_logs_path)
)

print("✅ dq_monitoring_logs carregada com sucesso")
print("Total de registros:", df_dq_logs.count())

print("\nSchema:")
df_dq_logs.printSchema()

print("\nAmostra dos dados:")
display(df_dq_logs)

In [0]:
# ============================================================
# VALIDAÇÃO DOS LOGS DE QUALIDADE
# ============================================================

print("===== RESUMO DQ_MONITORING_LOGS =====")

print("Total de registros:", df_dq_logs.count())

print("\nTabelas encontradas:")
df_dq_logs.groupBy("tabela") \
    .count() \
    .orderBy("tabela") \
    .show(truncate=False)

print("\nStatus encontrados:")
df_dq_logs.groupBy("status") \
    .count() \
    .show()

print("\nRegras encontradas:")
df_dq_logs.select("regra") \
    .distinct() \
    .orderBy("regra") \
    .show(100, truncate=False)

print("\nIntervalo de execução:")
df_dq_logs.select(
    F.min("timestamp_execucao").alias("primeira_execucao"),
    F.max("timestamp_execucao").alias("ultima_execucao")
).show(truncate=False)

In [0]:
# ============================================================
# GOLD 1 - RESUMO DE QUALIDADE POR REGRA
# Percentual de falhas por regra por dia
# ============================================================

gold_dq_resumo_por_regra = (
    df_dq_logs
    .withColumn(
        "data_execucao",
        F.to_date(F.col("timestamp_execucao"))
    )
    .groupBy(
        "data_execucao",
        "tabela",
        "regra"
    )
    .agg(
        F.sum("qtd_registros_falhos").alias("qtd_registros_falhos"),
        F.sum("qtd_registros_total").alias("qtd_registros_total")
    )
    .withColumn(
        "percentual_falhas",
        F.when(
            F.col("qtd_registros_total") > 0,
            F.round(
                (F.col("qtd_registros_falhos") /
                 F.col("qtd_registros_total")) * 100,
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .orderBy("data_execucao", "tabela", "regra")
)

print("✅ Gold - resumo por regra criado")
print("Total de registros:", gold_dq_resumo_por_regra.count())

display(gold_dq_resumo_por_regra)

In [0]:
# ============================================================
# VALIDAÇÃO DOS LOGS POR LOTE / ARQUIVO
# Antes de gerar o resumo por tabela
# ============================================================

df_validacao_logs = (
    df_dq_logs
    .select(
        "run_id",
        "tabela",
        "regra",
        "qtd_registros_falhos",
        "qtd_registros_total",
        "timestamp_execucao",
        "arquivo_origem"
    )
    .orderBy(
        "tabela",
        "timestamp_execucao",
        "arquivo_origem",
        "regra"
    )
)

print("✅ Logs preparados para validação")
print("Total:", df_validacao_logs.count())

display(df_validacao_logs)

In [0]:
# ============================================================
# VALIDAR GRANULARIDADE DOS LOGS
# Quantas regras existem para cada arquivo/lote
# ============================================================

df_lotes = (
    df_dq_logs
    .groupBy(
        "run_id",
        "tabela",
        "arquivo_origem"
    )
    .agg(
        F.countDistinct("regra").alias("qtd_regras"),
        F.max("qtd_registros_total").alias("registros_lote"),
        F.sum("qtd_registros_falhos").alias("falhas_somadas"),
        F.min("timestamp_execucao").alias("timestamp_execucao")
    )
    .orderBy("tabela", "timestamp_execucao")
)

print("===== LOTES / ARQUIVOS IDENTIFICADOS =====")
print("Total de lotes:", df_lotes.count())

display(df_lotes)

In [0]:
# ============================================================
# VERIFICAR LOTES DA ECOMMERCE_CLIENTES
# ============================================================

df_lotes_clientes = (
    df_lotes
    .filter(F.col("tabela") == "ecommerce_clientes")
    .orderBy("timestamp_execucao")
)

print("===== LOTES ECOMMERCE_CLIENTES =====")
print("Total de lotes:", df_lotes_clientes.count())

display(df_lotes_clientes)

In [0]:
# ============================================================
# DETALHAR ARQUIVOS DA ECOMMERCE_CLIENTES
# ============================================================

df_arquivos_clientes = (
    df_dq_logs
    .filter(F.col("tabela") == "ecommerce_clientes")
    .select(
        "run_id",
        "tabela",
        "regra",
        "status",
        "severidade",
        "qtd_registros_falhos",
        "qtd_registros_total",
        "timestamp_execucao",
        "arquivo_origem"
    )
    .orderBy("arquivo_origem", "regra")
)

print("===== ARQUIVOS ECOMMERCE_CLIENTES =====")

print(
    "Quantidade de arquivos distintos:",
    df_arquivos_clientes
        .select("arquivo_origem")
        .distinct()
        .count()
)

display(df_arquivos_clientes)

In [0]:
# ============================================================
# GOLD - CARREGAR SILVER ECOMMERCE_CLIENTES
# Fonte para os KPIs
# ============================================================

silver_clientes_path = (
    "abfss://squad1@internshipdatalake.dfs.core.windows.net/"
    "silver/ecommerce_clientes"
)

print("Caminho da Silver:")
print(silver_clientes_path)

df_clientes_silver = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(silver_clientes_path)
)

print("\n✅ Silver ecommerce_clientes carregada")
print("Total de registros:", df_clientes_silver.count())

print("\nColunas disponíveis:")
print(df_clientes_silver.columns)

In [0]:
# ============================================================
# VALIDAÇÃO DOS CADASTROS - 10 REGRAS DA SILVER
# ============================================================

from pyspark.sql import functions as F

print("===== VALIDAÇÃO DOS CADASTROS =====")

# Silver atualizada já contém as 10 flags
df_validacao = df_clientes_silver

# Lista das 10 regras
regras = [f"regra_{i:02d}_falhou" for i in range(1, 11)]

# Converte cada regra:
# False = 0 (não falhou)
# True  = 1 (falhou)
for regra in regras:
    df_validacao = df_validacao.withColumn(
        f"falha_{regra}",
        F.when(F.col(regra) == True, 1).otherwise(0)
    )

# Soma a quantidade de falhas por cliente
df_validacao = df_validacao.withColumn(
    "total_falhas",
    sum(F.col(f"falha_{regra}") for regra in regras)
)

# Classificação final
df_validacao = df_validacao.withColumn(
    "situacao_cadastro",
    F.when(
        F.col("total_falhas") == 0,
        "LIMPO"
    ).otherwise("COM_FALHA")
)

# Visualização
display(
    df_validacao.select(
        "id_cliente",
        *[f"falha_{regra}" for regra in regras],
        "total_falhas",
        "situacao_cadastro"
    )
)

In [0]:
resumo_limpeza = (
    df_validacao
    .agg(
        F.count("*").alias("total_cadastros"),
        F.sum(
            F.when(F.col("total_falhas") == 0, 1)
             .otherwise(0)
        ).alias("cadastros_limpos"),
        F.sum(
            F.when(F.col("total_falhas") >= 1, 1)
             .otherwise(0)
        ).alias("cadastros_com_falha")
    )
    .withColumn(
        "percentual_limpos",
        F.round(
            F.col("cadastros_limpos") * 100 /
            F.col("total_cadastros"), 2
        )
    )
)

display(resumo_limpeza)

In [0]:
display(
    df_validacao
    .groupBy("total_falhas")
    .count()
    .orderBy("total_falhas")
)

In [0]:
# ============================================================
# RESUMO DE REGISTROS LIMPOS POR DATA/HORA DO MICRO-LOTE
# Preparação e conferência 
# ============================================================

from pyspark.sql import functions as F
from functools import reduce

regras = [f"regra_{i:02d}_falhou" for i in range(1, 11)]

# Conferir as colunas e os tipos
tipos = dict(df_clientes_silver.dtypes)

invalidas = [
    regra for regra in regras
    if tipos.get(regra) != "boolean"
]

if invalidas:
    raise ValueError(
        f"Flags ausentes ou não booleanas: {invalidas}"
    )

tem_flag_nula = reduce(
    lambda a, b: a | b,
    [F.col(regra).isNull() for regra in regras]
)

if df_clientes_silver.filter(tem_flag_nula).limit(1).count():
    raise ValueError(
        "Existem flags nulas. Confira antes de calcular os limpos."
    )

# Extrair data/hora do caminho:
# real-time-data/AAAA/MM/DD/HHMMSS/...
padrao = r"real-time-data/(\d{4}/\d{2}/\d{2}/\d{6})/"

df_base_limpeza = df_clientes_silver.withColumn(
    "_lote_texto",
    F.regexp_extract("bronze_source_file", padrao, 1)
)

origem_invalida = (
    F.col("_lote_texto").isNull()
    | (F.col("_lote_texto") == "")
)

if df_base_limpeza.filter(origem_invalida).limit(1).count():
    raise ValueError(
        "Há arquivos sem data/hora no formato esperado."
    )

# Limpo somente quando as dez regras são False
passou_em_todas = reduce(
    lambda a, b: a & b,
    [F.col(regra) == F.lit(False) for regra in regras]
)

df_base_limpeza = (
    df_base_limpeza
    .withColumn(
        "_timestamp_lote",
        F.to_timestamp("_lote_texto", "yyyy/MM/dd/HHmmss")
    )
    .withColumn("_data", F.to_date("_timestamp_lote"))
    .withColumn(
        "_hora",
        F.date_format("_timestamp_lote", "HH:mm:ss")
    )
    .withColumn("_limpo", passou_em_todas.cast("long"))
)

if df_base_limpeza.filter(
    F.col("_timestamp_lote").isNull()
).limit(1).count():
    raise ValueError("Há datas/horas de lote inválidas.")

# Formato solicitado pelo avaliador
df_resumo_limpeza_gold = (
    df_base_limpeza
    .groupBy("_data", "_hora")
    .agg(
        F.sum("_limpo").alias("qtd_limpos"),
        F.count("*").alias("qtd_total")
    )
    .withColumn("tabela", F.lit("ecommerce_clientes"))
    .withColumn(
        "pct_limpo",
        F.round(
            F.col("qtd_limpos") * 100.0 / F.col("qtd_total"),
            2
        )
    )
    .select(
        "_data", "_hora", "qtd_limpos",
        "qtd_total", "tabela", "pct_limpo"
    )
)

display(df_resumo_limpeza_gold.orderBy("_data", "_hora"))

# Conferir os totais sem fazer média dos percentuais
display(
    df_resumo_limpeza_gold
    .agg(
        F.sum("qtd_total").alias("qtd_total"),
        F.sum("qtd_limpos").alias("qtd_limpos")
    )
    .withColumn(
        "pct_limpo",
        F.round(
            F.col("qtd_limpos") * 100.0 / F.col("qtd_total"),
            2
        )
    )
)

print("Resumo preparado. Nenhum dado foi gravado.")

In [0]:
delta_storage_options = {
    "AZURE_CLIENT_ID": client_id,
    "AZURE_TENANT_ID": tenant_id,
    "AZURE_CLIENT_SECRET": client_secret,
    "AZURE_STORAGE_ACCOUNT_NAME": storage_account
}

print("delta_storage_options criada com sucesso!")

In [0]:
from deltalake import write_deltalake

# ============================================================
# GRAVAR RESUMO DE LIMPEZA DOS CLIENTES NA GOLD
# ============================================================

from deltalake import write_deltalake, DeltaTable
import pyarrow as pa

gold_limpeza_path = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/"
    "gold/dq_resumo_limpeza_clientes"
)

colunas = [
    "_data", "_hora", "qtd_limpos",
    "qtd_total", "tabela", "pct_limpo"
]

dados = df_resumo_limpeza_gold.select(*colunas).toPandas()

# Conferir antes da gravação
if dados.empty:
    raise ValueError("O resumo está vazio.")

if dados[colunas].isna().any().any():
    raise ValueError("Existem valores nulos no resumo.")

if dados.duplicated(["tabela", "_data", "_hora"]).any():
    raise ValueError("Existem períodos duplicados.")

if (
    (dados["qtd_total"] <= 0)
    | (dados["qtd_limpos"] < 0)
    | (dados["qtd_limpos"] > dados["qtd_total"])
).any():
    raise ValueError("Existem contagens inválidas.")

total_silver = df_clientes_silver.count()

if int(dados["qtd_total"].sum()) != total_silver:
    raise ValueError("O total do resumo difere da Silver.")

# Tipos explícitos para gravação
schema = pa.schema([
    ("_data", pa.date32()),
    ("_hora", pa.string()),
    ("qtd_limpos", pa.int64()),
    ("qtd_total", pa.int64()),
    ("tabela", pa.string()),
    ("pct_limpo", pa.float64()),
])

dados_arrow = pa.Table.from_pandas(
    dados,
    schema=schema,
    preserve_index=False
)

write_deltalake(
    gold_limpeza_path,
    dados_arrow,
    mode="overwrite",
    schema_mode="overwrite",
    storage_options=delta_storage_options
)

# Reler e comparar todo o conteúdo gravado
gravado = DeltaTable(
    gold_limpeza_path,
    storage_options=delta_storage_options
).to_pandas()

chaves = ["tabela", "_data", "_hora"]

import pandas as pd

pd.testing.assert_frame_equal(
    dados.sort_values(chaves).reset_index(drop=True),
    gravado[colunas].sort_values(chaves).reset_index(drop=True),
    check_dtype=False
)

total = int(gravado["qtd_total"].sum())
limpos = int(gravado["qtd_limpos"].sum())

print("Resumo gravado e conferido no Delta.")
print("Períodos:", len(gravado))
print("Total de registros:", total)
print("Registros limpos:", limpos)
print("Percentual limpo:", round(limpos * 100.0 / total, 2))

display(gravado.sort_values(["_data", "_hora"]))

In [0]:
from deltalake import DeltaTable

# Ler a tabela Gold que acabamos de salvar
tabela_gold_limpeza = DeltaTable(
    gold_limpeza_path,
    storage_options=delta_storage_options
)

# Converter para Pandas
df_validacao_gold = tabela_gold_limpeza.to_pandas()

# Mostrar resultado
display(df_validacao_gold)

In [0]:
# ============================================================
# GOLD - REGRA 8
# VALIDAÇÃO DOS DADOS DA SILVER
# ============================================================

from pyspark.sql import functions as F

print("===== VALIDAÇÃO REGRA 8 =====")

# Total de registros
total_registros = df_clientes_silver.count()

# Total de clientes distintos
clientes_distintos = (
    df_clientes_silver
    .select("id_cliente")
    .distinct()
    .count()
)

print("Total de registros:", total_registros)
print("Clientes distintos:", clientes_distintos)


# ============================================================
# DISTRIBUIÇÃO DA REGRA 8
# False = passou na regra
# True  = falhou na regra
# ============================================================

print("\nDistribuição da regra_08_falhou:")

df_regra8_resumo = (
    df_clientes_silver
    .groupBy("regra_08_falhou")
    .agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("id_cliente").alias("clientes_distintos")
    )
    .orderBy("regra_08_falhou")
)

display(df_regra8_resumo)


# ============================================================
# REGISTROS QUE FALHARAM NA REGRA 8
# ============================================================

df_falhas_regra8 = (
    df_clientes_silver
    .filter(F.col("regra_08_falhou") == True)
    .select(
        "id_cliente",
        "regra_08_falhou"
    )
)

qtd_falhas_regra8 = df_falhas_regra8.count()

print(
    "\nTotal de registros que falharam na Regra 8:",
    qtd_falhas_regra8
)

display(df_falhas_regra8)

In [0]:
from pyspark.sql import functions as F

print("===== FALHAS DA REGRA 8 =====")

# Usa diretamente a flag persistida na Silver
df_regra8 = df_clientes_silver.select(
    "id_cliente",
    "regra_08_falhou"
)

# Resumo
df_cruz = (
    df_regra8
    .groupBy("regra_08_falhou")
    .agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("id_cliente").alias("clientes_distintos")
    )
    .orderBy("regra_08_falhou")
)

display(df_cruz)

# Clientes que falharam na Regra 8
df_falhas_regra8 = (
    df_regra8
    .filter(F.col("regra_08_falhou") == True)
)

total_falhas_geral = df_falhas_regra8.count()
clientes_com_falha = (
    df_falhas_regra8
    .select("id_cliente")
    .distinct()
    .count()
)

print("\nClientes/registros com falha na Regra 8:")
print("Registros com falha:", total_falhas_geral)
print("Clientes distintos com falha:", clientes_com_falha)

In [0]:
from pyspark.sql import functions as F

print("===== KPI GOLD - REGRA 8 =====")

gold_kpi_regra8 = (
    df_clientes_silver
    .agg(
        F.count("*").alias("total_registros"),

        F.countDistinct("id_cliente").alias("total_clientes"),

        F.sum(
            F.when(F.col("regra_08_falhou") == True, 1)
             .otherwise(0)
        ).alias("registros_com_falha"),

        F.countDistinct(
            F.when(
                F.col("regra_08_falhou") == True,
                F.col("id_cliente")
            )
        ).alias("clientes_com_falha")
    )
    .withColumn(
        "percentual_com_falha",
        F.round(
            F.col("registros_com_falha") * 100.0 /
            F.col("total_registros"),
            2
        )
    )
    .withColumn(
        "percentual_sem_falha",
        F.round(
            (F.col("total_registros") - F.col("registros_com_falha"))
            * 100.0 / F.col("total_registros"),
            2
        )
    )
)

display(gold_kpi_regra8)

In [0]:
# ============================================================
# REGRA 08 — RESUMO POR ARQUIVO COM CAMINHO RELATIVO
# ============================================================

from pyspark.sql import functions as F

if dict(df_clientes_silver.dtypes).get("regra_08_falhou") != "boolean":
    raise ValueError("regra_08_falhou precisa ser booleana.")

df_base_regra08 = df_clientes_silver.withColumn(
    "arquivo_origem",
    F.regexp_extract(
        F.col("bronze_source_file"),
        r"(?:^|/)(real-time-data/\d{4}/\d{2}/\d{2}/\d{6}/.+)$",
        1
    )
)

invalidos = df_base_regra08.filter(
    F.col("arquivo_origem").isNull()
    | (F.col("arquivo_origem") == "")
    | F.col("regra_08_falhou").isNull()
)

if invalidos.limit(1).count() > 0:
    display(
        invalidos.select(
            "bronze_source_file", "regra_08_falhou"
        ).limit(10)
    )
    raise ValueError("Há caminhos inválidos ou flags nulas.")

# Conta registros da Silver, preservando inclusive IDs repetidos.
# True = sem pedido; False = com pedido.
gold_regra8_por_lote = (
    df_base_regra08
    .groupBy("arquivo_origem")
    .agg(
        F.count("*").alias("total_clientes"),
        F.sum(
            F.when(~F.col("regra_08_falhou"), 1).otherwise(0)
        ).alias("clientes_com_pedido"),
        F.sum(
            F.when(F.col("regra_08_falhou"), 1).otherwise(0)
        ).alias("clientes_sem_pedido")
    )
    .withColumn(
        "percentual_com_pedido",
        F.round(
            100.0 * F.col("clientes_com_pedido")
            / F.col("total_clientes"), 2
        )
    )
    .withColumn(
        "percentual_sem_pedido",
        F.round(
            100.0 * F.col("clientes_sem_pedido")
            / F.col("total_clientes"), 2
        )
    )
    .withColumn("clientes_sem_status", F.lit(0).cast("long"))
    .select(
        "arquivo_origem",
        "total_clientes",
        "clientes_com_pedido",
        "clientes_sem_pedido",
        "percentual_com_pedido",
        "percentual_sem_pedido",
        "clientes_sem_status"
    )
)

display(gold_regra8_por_lote.orderBy("arquivo_origem"))

print("Resumo preparado. Nenhum dado foi gravado.")

In [0]:
# ============================================================
# VALIDAR O RESUMO ATUALIZADO DA REGRA 08
# ============================================================

total_lotes = gold_regra8_por_lote.count()

duplicados = (
    gold_regra8_por_lote
    .groupBy("arquivo_origem")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

inconsistentes = gold_regra8_por_lote.filter(
    (F.col("total_clientes") <= 0)
    | (
        F.col("total_clientes")
        != (
            F.col("clientes_com_pedido")
            + F.col("clientes_sem_pedido")
            + F.col("clientes_sem_status")
        )
    )
).count()

resumo = gold_regra8_por_lote.agg(
    F.sum("total_clientes").alias("total"),
    F.sum("clientes_com_pedido").alias("com_pedido"),
    F.sum("clientes_sem_pedido").alias("sem_pedido"),
    F.sum("clientes_sem_status").alias("sem_status")
).first()

total_silver = df_clientes_silver.count()
falhas_silver = df_clientes_silver.filter(
    F.col("regra_08_falhou") == True
).count()

if total_lotes == 0 or duplicados or inconsistentes:
    raise ValueError("Resumo vazio, duplicado ou inconsistente.")

if resumo["total"] != total_silver:
    raise ValueError("O total do resumo diverge da Silver.")

if resumo["sem_pedido"] != falhas_silver:
    raise ValueError("A quantidade de falhas diverge da Silver.")

print("VALIDAÇÃO APROVADA — RESUMO DA REGRA 08")
print("Micro-lotes:", total_lotes)
print("Total de registros:", resumo["total"])
print("Com pedido:", resumo["com_pedido"])
print("Sem pedido:", resumo["sem_pedido"])
print("Sem status:", resumo["sem_status"])
print("Arquivos duplicados:", duplicados)
print("Nenhum dado foi gravado.")

In [0]:
# ============================================================
# PREPARAÇÃO DO DELTA — REGRA 08
# Suporta primeira carga e atualização.
# ============================================================

from pyspark.sql import functions as F
from pyspark.errors import AnalysisException


def preparar_publicacao_clientes(df_resumo, caminho):
    # Materializa os dados para gravação e comparação.
    linhas = df_resumo.collect()

    if not linhas:
        raise ValueError("Resumo vazio. Publicação cancelada.")

    df_publicacao = spark.createDataFrame(
        linhas,
        schema=df_resumo.schema
    )

    padrao = (
        r"^real-time-data/\d{4}/\d{2}/\d{2}/\d{6}/.+$"
    )

    if df_publicacao.filter(
        F.col("arquivo_origem").isNull()
        | ~F.col("arquivo_origem").rlike(padrao)
    ).limit(1).count():
        raise ValueError(
            "O resumo contém caminhos ausentes ou fora do padrão."
        )

    duplicados = (
        df_publicacao
        .groupBy("arquivo_origem")
        .count()
        .filter(F.col("count") > 1)
    )

    if duplicados.limit(1).count():
        raise ValueError(
            "O resumo contém mais de uma linha por arquivo."
        )

    try:
        df_anterior = (
            spark.read
            .format("delta")
            .options(**adls_options)
            .load(caminho)
        )

        # Força a leitura dentro do tratamento de erros.
        df_anterior.limit(1).collect()

    except AnalysisException as erro:
        if erro.getErrorClass() != "PATH_NOT_FOUND":
            raise

        print("Primeira carga: caminho de destino ainda não existe.")
        df_anterior = None

    if df_anterior is not None:
        arquivos_anteriores = (
            df_anterior
            .select(
                F.regexp_extract(
                    F.col("arquivo_origem"),
                    (
                        r"(?:^|/)(real-time-data/"
                        r"\d{4}/\d{2}/\d{2}/\d{6}/.+)$"
                    ),
                    1
                ).alias("arquivo_origem")
            )
            .distinct()
        )

        if arquivos_anteriores.filter(
            F.col("arquivo_origem").isNull()
            | (F.col("arquivo_origem") == "")
        ).limit(1).count():
            raise ValueError(
                "O Delta existente contém caminhos inválidos."
            )

        arquivos_ausentes = arquivos_anteriores.join(
            df_publicacao.select("arquivo_origem"),
            on="arquivo_origem",
            how="left_anti"
        )

        if arquivos_ausentes.limit(1).count():
            display(arquivos_ausentes)
            raise ValueError(
                "O novo resumo não contempla todos os arquivos "
                "do Delta existente. Publicação cancelada."
            )

        print("Todos os arquivos existentes estão contemplados.")

    print("PREPARAÇÃO APROVADA")
    print("Lotes preparados:", len(linhas))
    print("Nenhum dado foi gravado.")

    return df_publicacao


gold_regra8_path = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/"
    "gold/ecommerce_clientes/regra_08_clientes_pedidos"
)

df_regra08_delta_publicacao = preparar_publicacao_clientes(
    gold_regra8_por_lote,
    gold_regra8_path
)

In [0]:
# ============================================================
# GRAVAR E VALIDAR O DELTA DA REGRA 08 USANDO SPARK
# ============================================================

(
    df_regra08_delta_publicacao.write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_regra8_path)
)

# Relê o conteúdo efetivamente persistido.
df_regra08_delta_validacao = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(gold_regra8_path)
    .select(*df_regra08_delta_publicacao.columns)
)

faltando = (
    df_regra08_delta_publicacao
    .exceptAll(df_regra08_delta_validacao)
    .limit(1)
    .count()
)

sobrando = (
    df_regra08_delta_validacao
    .exceptAll(df_regra08_delta_publicacao)
    .limit(1)
    .count()
)

if faltando or sobrando:
    raise ValueError(
        "A gravação terminou, mas o Delta diverge do resumo preparado."
    )

print("VALIDAÇÃO APROVADA: Delta corresponde ao resumo preparado.")
print("Lotes gravados:", df_regra08_delta_validacao.count())
print("arquivo_origem: caminho relativo.")

display(
    df_regra08_delta_validacao.agg(
        F.sum("total_clientes").alias("total_registros"),
        F.sum("clientes_com_pedido").alias("com_pedido"),
        F.sum("clientes_sem_pedido").alias("sem_pedido"),
        F.sum("clientes_sem_status").alias("sem_status")
    )
)

In [0]:
# ============================================================
# GOLD - REGRA 8
# VALIDACAO FINAL DA GRAVACAO
# ============================================================

from pyspark.sql import functions as F

df_gold_regra8_salva = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(gold_regra8_path)
)

print("Total de lotes gravados:", df_gold_regra8_salva.count())

print(
    "Arquivos distintos:",
    df_gold_regra8_salva
    .select("arquivo_origem")
    .distinct()
    .count()
)

display(
    df_gold_regra8_salva.agg(
        F.sum("total_clientes").alias("total_clientes"),
        F.sum("clientes_com_pedido").alias("com_pedido"),
        F.sum("clientes_sem_pedido").alias("sem_pedido"),
        F.sum("clientes_sem_status").alias("sem_status")
    )
)

In [0]:
# ============================================================
# REGRA 09 — RECALCULAR OS MICRO-LOTES A PARTIR DA SILVER
# ============================================================

from pyspark.sql import functions as F

if dict(df_clientes_silver.dtypes).get("regra_09_falhou") != "boolean":
    raise ValueError("regra_09_falhou precisa ser booleana.")

df_base_regra09 = (
    df_clientes_silver
    .withColumn(
        "arquivo_origem",
        F.regexp_extract(
            F.col("bronze_source_file"),
            r"(?:^|/)(real-time-data/\d{4}/\d{2}/\d{2}/\d{6}/.+)$",
            1
        )
    )
)

invalidos = df_base_regra09.filter(
    F.col("arquivo_origem").isNull()
    | (F.col("arquivo_origem") == "")
    | F.col("regra_09_falhou").isNull()
)

if invalidos.limit(1).count() > 0:
    display(
        invalidos.select(
            "bronze_source_file", "regra_09_falhou"
        ).limit(10)
    )
    raise ValueError("Há caminhos inválidos ou flags nulas.")

df_base_regra09 = df_base_regra09.withColumn(
    "_timestamp_lote",
    F.to_timestamp(
        F.regexp_extract(
            "arquivo_origem",
            r"^real-time-data/(\d{4}/\d{2}/\d{2}/\d{6})/",
            1
        ),
        "yyyy/MM/dd/HHmmss"
    )
)

if df_base_regra09.filter(
    F.col("_timestamp_lote").isNull()
).limit(1).count():
    raise ValueError("Há datas de micro-lote inválidas.")

gold_regra_09_por_lote = (
    df_base_regra09
    .groupBy("arquivo_origem", "_timestamp_lote")
    .agg(
        F.count("*").alias("total_clientes"),
        F.sum(
            F.when(~F.col("regra_09_falhou"), 1).otherwise(0)
        ).alias("emails_provedores_reconhecidos")
    )
    .withColumn("data_lote", F.to_date("_timestamp_lote"))
    .withColumn(
        "hora_lote",
        F.date_format("_timestamp_lote", "HHmmss")
    )
    .withColumn(
        "percentual_provedores_reconhecidos",
        F.round(
            100.0 * F.col("emails_provedores_reconhecidos")
            / F.col("total_clientes"),
            2
        )
    )
    .withColumn("meta_percentual", F.lit(95.0))
    # Avalia a meta antes do arredondamento do percentual.
    .withColumn(
        "status_kpi",
        F.when(
            F.col("emails_provedores_reconhecidos") * 100
            >= F.col("total_clientes") * 95,
            "PASS"
        ).otherwise("FAIL")
    )
    .drop("_timestamp_lote")
)

gold_indicadores_clientes = (
    gold_regra_09_por_lote
    .select(
        "data_lote",
        "hora_lote",
        "arquivo_origem",
        "total_clientes",
        "emails_provedores_reconhecidos",
        "percentual_provedores_reconhecidos",
        "meta_percentual",
        F.col("status_kpi").alias("status_regra_09")
    )
    .withColumn(
        "qtd_emails_nao_reconhecidos",
        F.col("total_clientes")
        - F.col("emails_provedores_reconhecidos")
    )
)

resumo_regra09 = gold_indicadores_clientes.agg(
    F.count("*").alias("lotes"),
    F.countDistinct("arquivo_origem").alias("arquivos"),
    F.sum("total_clientes").alias("total"),
    F.sum("emails_provedores_reconhecidos").alias("reconhecidos"),
    F.sum("qtd_emails_nao_reconhecidos").alias("falhas")
).first()

referencia_regra09 = df_clientes_silver.agg(
    F.count("*").alias("total"),
    F.sum(F.col("regra_09_falhou").cast("long")).alias("falhas")
).first()

if (
    resumo_regra09["lotes"] == 0
    or resumo_regra09["lotes"] != resumo_regra09["arquivos"]
    or resumo_regra09["total"] != referencia_regra09["total"]
    or resumo_regra09["falhas"] != referencia_regra09["falhas"]
):
    raise ValueError("O resumo está vazio, duplicado ou diverge da Silver.")

print("VALIDAÇÃO APROVADA — PREPARAÇÃO DA REGRA 09")
print("Micro-lotes:", resumo_regra09["lotes"])
print("Total de registros:", resumo_regra09["total"])
print("E-mails reconhecidos:", resumo_regra09["reconhecidos"])
print("E-mails não reconhecidos:", resumo_regra09["falhas"])
print("Nenhum dado foi gravado.")

In [0]:
# ============================================================
# GOLD - REGRA 9
# TABELA FINAL KPI PROVEDORES DE EMAIL
# ============================================================

from pyspark.sql import functions as F

gold_kpi_email_provedores = (
    gold_regra_09_por_lote
    .select(
        "data_lote",
        "hora_lote",
        F.lit("ecommerce_clientes").alias("tabela"),
        F.lit("regra_09_provedores_email").alias("regra"),
        "total_clientes",
        "emails_provedores_reconhecidos",
        "percentual_provedores_reconhecidos",
        "meta_percentual",
        "status_kpi",
        "arquivo_origem"
    )
    .orderBy("data_lote", "hora_lote")
)

print("===== GOLD KPI REGRA 9 =====")
print("Total de lotes:", gold_kpi_email_provedores.count())

display(gold_kpi_email_provedores)

In [0]:
# ============================================================
# PREPARAÇÃO DO DELTA — REGRA 09
# ============================================================

gold_indicadores_path = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/"
    "gold/ecommerce_clientes/indicadores"
)

df_regra09_delta_publicacao = preparar_publicacao_clientes(
    gold_indicadores_clientes,
    gold_indicadores_path
)

In [0]:
# ============================================================
# GRAVAR E VALIDAR O DELTA ATUALIZADO — REGRA 09
# ============================================================

(
    df_regra09_delta_publicacao.write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_indicadores_path)
)

df_regra09_delta_validacao = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(gold_indicadores_path)
    .select(*df_regra09_delta_publicacao.columns)
)

faltando_regra09 = (
    df_regra09_delta_publicacao
    .exceptAll(df_regra09_delta_validacao)
    .limit(1)
    .count()
)

sobrando_regra09 = (
    df_regra09_delta_validacao
    .exceptAll(df_regra09_delta_publicacao)
    .limit(1)
    .count()
)

if faltando_regra09 or sobrando_regra09:
    raise ValueError(
        "A gravação terminou, mas o Delta diverge do resumo preparado."
    )

print("VALIDAÇÃO APROVADA: Delta corresponde ao resumo da regra 09.")
print("Lotes gravados:", df_regra09_delta_validacao.count())
print("arquivo_origem: caminho relativo.")

display(
    df_regra09_delta_validacao.agg(
        F.sum("total_clientes").alias("total_registros"),
        F.sum("emails_provedores_reconhecidos").alias(
            "emails_reconhecidos"
        ),
        F.sum("qtd_emails_nao_reconhecidos").alias(
            "emails_nao_reconhecidos"
        )
    )
)

In [0]:
# ============================================================
# VALIDACAO FINAL DA GRAVACAO
# ============================================================

from pyspark.sql import functions as F

df_gold_validacao = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(gold_indicadores_path)
)

print("===== VALIDACAO FINAL GOLD =====")

print("Total de lotes gravados:", df_gold_validacao.count())

print(
    "Arquivos distintos:",
    df_gold_validacao
    .select("arquivo_origem")
    .distinct()
    .count()
)

display(
    df_gold_validacao
    .groupBy("status_regra_09")
    .count()
    .orderBy("status_regra_09")
)

In [0]:
# Verificar duplicidades antes do dashboard

from pyspark.sql import functions as F

df_gold_check = df_gold_validacao.withColumn(
    "arquivo_normalizado",
    F.regexp_extract(
        F.col("arquivo_origem"),
        r"(real-time-data/.*)",
        1
    )
)

df_duplicados = (
    df_gold_check
    .filter(F.col("arquivo_normalizado") != "")
    .groupBy("arquivo_normalizado")
    .count()
    .filter(F.col("count") > 1)
)

print("Total Gold:", df_gold_check.count())

print(
    "Arquivos normalizados distintos:",
    df_gold_check
    .select("arquivo_normalizado")
    .distinct()
    .count()
)

print("Arquivos duplicados:", df_duplicados.count())

display(df_duplicados)

In [0]:
# Preparar indicadores sem duplicidade para o dashboard

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Prioriza o registro com caminho completo
janela = Window.partitionBy(
    "arquivo_normalizado"
).orderBy(
    F.length(F.col("arquivo_origem")).desc()
)

df_gold_dashboard = (
    df_gold_check
    .withColumn(
        "ordem",
        F.row_number().over(janela)
    )
    .filter(F.col("ordem") == 1)
    .drop("ordem")
)

print("===== GOLD PARA DASHBOARD =====")
print("Registros originais:", df_gold_check.count())
print("Registros sem duplicidade:", df_gold_dashboard.count())

display(
    df_gold_dashboard.groupBy("status_regra_09").count()
)

In [0]:
# Conferir se os lotes duplicados têm resultados diferentes

from pyspark.sql import functions as F

df_conflitos = (
    df_gold_check
    .groupBy("arquivo_normalizado")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct(
            "status_regra_09",
            "total_clientes",
            "percentual_provedores_reconhecidos"
        ).alias("versoes_indicadores")
    )
    .filter(
        (F.col("registros") > 1) &
        (F.col("versoes_indicadores") > 1)
    )
)

print("===== VALIDAÇÃO DOS INDICADORES =====")
print("Arquivos com indicadores diferentes:", df_conflitos.count())

display(df_conflitos)

In [0]:
# KPI por lote para o Looker Studio

from pyspark.sql import functions as F

gold_clientes_kpi_lote = (
    df_gold_dashboard
    .select(
        F.col("arquivo_normalizado").alias("arquivo_origem"),
        F.col("data_lote"),
        F.col("hora_lote"),
        F.col("total_clientes"),
        F.col("emails_provedores_reconhecidos"),
        F.col("percentual_provedores_reconhecidos"),
        F.col("meta_percentual"),
        F.col("status_regra_09"),
        F.col("qtd_emails_nao_reconhecidos")
    )
)

print("===== KPI GOLD POR LOTE =====")
print("Total de lotes:", gold_clientes_kpi_lote.count())

display(gold_clientes_kpi_lote.limit(10))

In [0]:
# Leitura dos logs de qualidade

dq_logs_path = (
    "abfss://squad1@internshipdatalake"
    ".dfs.core.windows.net/dq_monitoring_logs"
)

df_dq_clientes = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(dq_logs_path)
    .filter(F.col("tabela") == "ecommerce_clientes")
)

print("===== LOGS DE QUALIDADE =====")
print("Total de logs:", df_dq_clientes.count())

display(
    df_dq_clientes.groupBy("regra")
    .count()
    .orderBy("regra")
)

In [0]:
# Detalhamento dos logs de qualidade

print("===== DETALHAMENTO DOS LOGS =====")

display(
    df_dq_clientes
    .select(
        "regra",
        "status",
        "severidade",
        "qtd_registros_falhos",
        "qtd_registros_total",
        "timestamp_execucao",
        "arquivo_origem"
    )
    .orderBy(F.col("timestamp_execucao").desc())
)

In [0]:
# ============================================================
# RESUMO DAS VALIDAÇÕES DA SILVER
# ============================================================

from pyspark.sql import functions as F
from functools import reduce

# As 10 regras persistidas na Silver
colunas_regras = [
    f"regra_{i:02d}_falhou"
    for i in range(1, 11)
]

resumos = []

for regra in colunas_regras:

    df_resumo = (
        df_clientes_silver
        .groupBy(
            F.col(regra).alias("falhou")
        )
        .agg(
            F.count("*").alias("quantidade")
        )
        .withColumn(
            "regra",
            F.lit(regra)
        )
        .select(
            "regra",
            "falhou",
            "quantidade"
        )
    )

    resumos.append(df_resumo)

# Junta o resultado das 10 regras
gold_resumo_validacoes = reduce(
    lambda a, b: a.unionByName(b),
    resumos
)

print("===== RESUMO DE QUALIDADE - SILVER =====")

display(
    gold_resumo_validacoes
    .orderBy("regra", "falhou")
)

In [0]:
# ============================================================
# KPIs DE QUALIDADE - REGRAS
# ============================================================

from pyspark.sql import functions as F

total_registros = df_clientes_silver.count()

gold_kpi_qualidade = (
    gold_resumo_validacoes
    .withColumn(
        "percentual",
        F.when(
            F.lit(total_registros) > 0,
            F.round(
                F.col("quantidade") * 100 / F.lit(total_registros),
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "total_registros",
        F.lit(total_registros)
    )
)

print("===== KPIs DE QUALIDADE =====")

display(
    gold_kpi_qualidade.orderBy("regra", "falhou")
)

In [0]:
# ============================================================
# KPIs DE QUALIDADE POR DIA DE CADASTRO
# Usa as flags booleanas da Silver
# ============================================================

from pyspark.sql import functions as F

# Regra de qualidade -> nome do indicador
indicadores = {
    "regra_04_falhou": "senhas_validas",
    "regra_05_falhou": "cadastros_validos",
    "regra_09_falhou": "provedores_validos",
    "regra_08_falhou": "clientes_com_pedido",
}

# 1. Conferir as colunas necessárias
necessarias = ["dt_cadastro", *indicadores.keys()]

faltantes = [
    coluna for coluna in necessarias
    if coluna not in df_clientes_silver.columns
]

if faltantes:
    raise ValueError(
        f"Colunas ausentes na Silver: {faltantes}\n"
        f"Colunas disponíveis: {df_clientes_silver.columns}"
    )

tipos = dict(df_clientes_silver.dtypes)

nao_booleanas = [
    coluna for coluna in indicadores
    if tipos[coluna] != "boolean"
]

if nao_booleanas:
    raise ValueError(
        f"As flags precisam ser booleanas: {nao_booleanas}"
    )

# 2. Contar registros aprovados em cada regra
# Apenas False conta como aprovado; nulos não contam.
agregacoes = [
    F.sum(
        F.when(F.col(coluna) == F.lit(False), 1)
         .otherwise(0)
    ).alias(indicador)
    for coluna, indicador in indicadores.items()
]

# 3. Agrupar por dia de cadastro
df_kpi_dia = (
    df_clientes_silver
    .withColumn(
        "data_referencia",
        F.to_date("dt_cadastro")
    )
    .groupBy("data_referencia")
    .agg(
        F.count("*").alias("total_clientes"),
        *agregacoes
    )
)

# 4. Exibir os resultados
print("===== KPIs POR DIA DE CADASTRO =====")

display(
    df_kpi_dia.orderBy("data_referencia")
)

In [0]:
# ============================================================
# GOLD - KPIs POR HORA DE INGESTÃO
# False na flag significa que o registro passou na regra
# ============================================================

from pyspark.sql import functions as F

df_kpi_hora = (
    df_clientes_silver
    .withColumn(
        "hora_referencia",
        F.date_trunc("hour", F.col("bronze_ingested_at"))
    )
    .groupBy("hora_referencia")
    .agg(
        F.count("*").alias("total_clientes"),

        F.sum(
            F.when(
                F.col("regra_04_falhou") == False, 1
            ).otherwise(0)
        ).alias("senhas_validas"),

        F.sum(
            F.when(
                F.col("regra_05_falhou") == False, 1
            ).otherwise(0)
        ).alias("cadastros_validos"),

        F.sum(
            F.when(
                F.col("regra_09_falhou") == False, 1
            ).otherwise(0)
        ).alias("provedores_validos")
    )
)

print("===== KPIs POR HORA DE INGESTÃO =====")

display(
    df_kpi_hora.orderBy(
        F.col("hora_referencia").desc()
    )
)

In [0]:
# ============================================================
# VALIDAÇÃO DOS KPIs POR HORA
# ============================================================

from pyspark.sql import functions as F

df_validacao_hora = df_kpi_hora.agg(
    F.sum("total_clientes").alias("total_clientes"),
    F.sum("senhas_validas").alias("senhas_validas"),
    F.sum("cadastros_validos").alias("cadastros_validos"),
    F.sum("provedores_validos").alias("provedores_validos")
)

print("===== VALIDAÇÃO DOS KPIs POR HORA =====")
display(df_validacao_hora)

total_silver = df_clientes_silver.count()
total_hora = df_validacao_hora.first()["total_clientes"]

print("Total Silver:", total_silver)
print("Total KPI por hora:", total_hora)

if total_silver == total_hora:
    print("VALIDAÇÃO APROVADA!")
else:
    raise ValueError("Divergência nos KPIs por hora!")

In [0]:
import os
from dotenv import load_dotenv

load_dotenv()

jdbc_hostname = os.getenv("SQL_HOST")
jdbc_database = os.getenv("SQL_DATABASE")
jdbc_username = os.getenv("SQL_USERNAME")
jdbc_password = os.getenv("SQL_PASSWORD")

# Validar configuração sem mostrar credenciais
variaveis = {
    "SQL_HOST": jdbc_hostname,
    "SQL_DATABASE": jdbc_database,
    "SQL_USERNAME": jdbc_username,
    "SQL_PASSWORD": jdbc_password
}

faltando = [nome for nome, valor in variaveis.items() if not valor]

if faltando:
    raise ValueError(
        "Configuração ausente: " + ", ".join(faltando)
    )

print("Configuração SQL carregada com sucesso!")

In [0]:
# TESTE DE CONEXÃO COM AZURE SQL

df_teste_sql = (
    spark.read
    .format("sqlserver")
    .option("host", jdbc_hostname)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("query", "SELECT 1 AS teste")
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("encrypt", "true")
    .load()
)

display(df_teste_sql)

In [0]:
# ============================================================
# PUBLICAR RESUMO DE LIMPEZA DOS CLIENTES NO AZURE SQL
# ============================================================

from pyspark.sql import functions as F

gold_limpeza_path = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/"
    "gold/dq_resumo_limpeza_clientes"
)

tabela_sql_limpeza = "squad1.gold_clientes_dq_tabela_hora"

colunas_resumo = [
    "_data",
    "_hora",
    "qtd_limpos",
    "qtd_total",
    "tabela",
    "pct_limpo",
]

# 1. Ler o resumo efetivamente gravado no Delta
df_publicacao_limpeza = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(gold_limpeza_path)
    .select(*colunas_resumo)
    .cache()
)

# 2. Conferir os dados antes da publicação
total_periodos = df_publicacao_limpeza.count()

if total_periodos == 0:
    raise ValueError("O resumo Delta está vazio.")

condicao_nulos = F.lit(False)

for coluna in colunas_resumo:
    condicao_nulos = condicao_nulos | F.col(coluna).isNull()

if df_publicacao_limpeza.filter(condicao_nulos).limit(1).count():
    raise ValueError("Existem valores nulos no resumo.")

duplicados = (
    df_publicacao_limpeza
    .groupBy("tabela", "_data", "_hora")
    .count()
    .filter(F.col("count") > 1)
)

if duplicados.limit(1).count():
    raise ValueError("Existem períodos duplicados.")

invalidos = df_publicacao_limpeza.filter(
    (F.col("tabela") != "ecommerce_clientes")
    | (F.col("qtd_total") <= 0)
    | (F.col("qtd_limpos") < 0)
    | (F.col("qtd_limpos") > F.col("qtd_total"))
    | (
        F.abs(
            F.col("pct_limpo")
            - F.round(
                F.col("qtd_limpos") * 100.0 / F.col("qtd_total"),
                2
            )
        ) > 0.000001
    )
)

if invalidos.limit(1).count():
    raise ValueError("Existem contagens ou percentuais inválidos.")

# 3. Configuração da conexão
opcoes_sql_limpeza = {
    "host": jdbc_hostname,
    "port": "1433",
    "database": jdbc_database,
    "dbtable": tabela_sql_limpeza,
    "user": jdbc_username,
    "password": jdbc_password,
    "encrypt": "true",
}

# 4. Publicar o resumo completo
# Atualiza também os períodos que já existiam.
(
    df_publicacao_limpeza.write
    .format("sqlserver")
    .options(**opcoes_sql_limpeza)
    .mode("overwrite")
    .save()
)

print("Publicação concluída:", tabela_sql_limpeza)
print("Períodos enviados:", total_periodos)

In [0]:
# ============================================================
# VALIDAR O RESUMO PUBLICADO NO AZURE SQL
# ============================================================

from pyspark.sql import functions as F

df_sql_gold = (
    spark.read
    .format("sqlserver")
    .options(**opcoes_sql_limpeza)
    .load()
    .select(*colunas_resumo)
)

# Alinhar os tipos retornados pelo SQL para comparar os dados
tipos_origem = dict(df_publicacao_limpeza.dtypes)

df_sql_comparacao = df_sql_gold.select(
    *[
        F.col(coluna).cast(tipos_origem[coluna]).alias(coluna)
        for coluna in colunas_resumo
    ]
)

faltando_no_sql = (
    df_publicacao_limpeza
    .exceptAll(df_sql_comparacao)
    .limit(1)
    .count()
)

sobrando_no_sql = (
    df_sql_comparacao
    .exceptAll(df_publicacao_limpeza)
    .limit(1)
    .count()
)

if faltando_no_sql or sobrando_no_sql:
    raise ValueError("O conteúdo do SQL difere do resumo Delta.")

print("VALIDAÇÃO APROVADA: SQL e Delta têm o mesmo conteúdo.")
print("Tabela:", tabela_sql_limpeza)
print("Períodos no SQL:", df_sql_gold.count())

display(
    df_sql_gold
    .agg(
        F.sum("qtd_total").alias("qtd_total"),
        F.sum("qtd_limpos").alias("qtd_limpos")
    )
    .withColumn(
        "pct_limpo",
        F.round(
            F.col("qtd_limpos") * 100.0 / F.col("qtd_total"),
            2
        )
    )
)

display(df_sql_gold.orderBy("_data", "_hora"))

df_publicacao_limpeza.unpersist()

In [0]:
# ============================================================
# PUBLICAR REGRA 08 ATUALIZADA NO SQL E VALIDAR
# ============================================================

from pyspark.sql import functions as F

gold_regra8_path = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/"
    "gold/ecommerce_clientes/regra_08_clientes_pedidos"
)

opcoes_sql_regra08 = {
    "host": jdbc_hostname,
    "port": "1433",
    "database": jdbc_database,
    "dbtable": "squad1.gold_clientes_regra08",
    "user": jdbc_username,
    "password": jdbc_password,
    "encrypt": "true"
}

# Lê o Delta atualizado.
df_regra08_origem = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(gold_regra8_path)
)

# Materializa os dados para publicar e comparar a mesma versão.
linhas_regra08 = df_regra08_origem.collect()

if not linhas_regra08:
    raise ValueError("Delta vazio. Publicação cancelada.")

df_regra08_publicacao_sql = spark.createDataFrame(
    linhas_regra08,
    schema=df_regra08_origem.schema
)

invalidos = df_regra08_publicacao_sql.filter(
    F.col("arquivo_origem").isNull()
    | ~F.col("arquivo_origem").rlike(
        r"^real-time-data/\d{4}/\d{2}/\d{2}/\d{6}/.+$"
    )
)

if invalidos.limit(1).count() > 0:
    raise ValueError("Há caminhos inválidos. Publicação cancelada.")

duplicados = (
    df_regra08_publicacao_sql
    .groupBy("arquivo_origem")
    .count()
    .filter(F.col("count") > 1)
)

if duplicados.limit(1).count() > 0:
    raise ValueError("Há arquivos duplicados. Publicação cancelada.")

# Atualiza a tabela completa, incluindo os lotes já existentes.
(
    df_regra08_publicacao_sql.write
    .format("sqlserver")
    .options(**opcoes_sql_regra08)
    .mode("overwrite")
    .save()
)

# Relê o SQL e alinha os tipos para comparar todas as colunas.
df_regra08_sql_validacao = (
    spark.read
    .format("sqlserver")
    .options(**opcoes_sql_regra08)
    .load()
)

df_regra08_sql_comparacao = df_regra08_sql_validacao.select(
    *[
        F.col(campo.name).cast(campo.dataType).alias(campo.name)
        for campo in df_regra08_publicacao_sql.schema.fields
    ]
)

faltando = (
    df_regra08_publicacao_sql
    .exceptAll(df_regra08_sql_comparacao)
    .limit(1)
    .count()
)

sobrando = (
    df_regra08_sql_comparacao
    .exceptAll(df_regra08_publicacao_sql)
    .limit(1)
    .count()
)

if faltando or sobrando:
    raise ValueError(
        "A gravação terminou, mas o SQL diverge do Delta."
    )

print("VALIDAÇÃO APROVADA: SQL corresponde ao Delta atualizado.")
print("Tabela:", opcoes_sql_regra08["dbtable"])
print("Lotes publicados:", len(linhas_regra08))
print("arquivo_origem: caminho relativo.")

display(
    df_regra08_sql_validacao.agg(
        F.sum("total_clientes").alias("total_registros"),
        F.sum("clientes_com_pedido").alias("com_pedido"),
        F.sum("clientes_sem_pedido").alias("sem_pedido"),
        F.sum("clientes_sem_status").alias("sem_status")
    )
)

In [0]:
# VALIDAR REGRA 8 NO AZURE SQL

df_sql_regra8 = (
    spark.read
    .format("sqlserver")
    .option("host", jdbc_hostname)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", "squad1.gold_clientes_regra08")
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("encrypt", "true")
    .load()
)

total = df_sql_regra8.count()
arquivos = df_sql_regra8.select(
    "arquivo_origem"
).distinct().count()

print("Total de registros:", total)
print("Arquivos distintos:", arquivos)
print("Duplicações:", total - arquivos)

display(df_sql_regra8.limit(10))

In [0]:
# ============================================================
# PUBLICAR E VALIDAR A REGRA 09 NO SQL
# ============================================================

from pyspark.sql import functions as F

gold_indicadores_path = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/"
    "gold/ecommerce_clientes/indicadores"
)

colunas_regra09 = [
    "arquivo_origem",
    "data_lote",
    "hora_lote",
    "total_clientes",
    "emails_provedores_reconhecidos",
    "percentual_provedores_reconhecidos",
    "meta_percentual",
    "status_regra_09",
    "qtd_emails_nao_reconhecidos"
]

opcoes_sql_regra09 = {
    "host": jdbc_hostname,
    "port": "1433",
    "database": jdbc_database,
    "dbtable": "squad1.gold_clientes_regra09",
    "user": jdbc_username,
    "password": jdbc_password,
    "encrypt": "true"
}

# Lê o Delta persistido.
df_regra09_origem = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(gold_indicadores_path)
    .select(*colunas_regra09)
)

# Materializa a versão que será publicada.
linhas_regra09_sql = df_regra09_origem.collect()

if not linhas_regra09_sql:
    raise ValueError("Delta vazio. Publicação cancelada.")

df_regra09_publicacao_sql = spark.createDataFrame(
    linhas_regra09_sql,
    schema=df_regra09_origem.schema
)

# Confere se o Delta contém o resumo atualizado da célula 34.
df_regra09_esperado = df_regra09_delta_publicacao.select(
    *colunas_regra09
)

if (
    df_regra09_esperado
    .exceptAll(df_regra09_publicacao_sql)
    .limit(1).count()
    or df_regra09_publicacao_sql
    .exceptAll(df_regra09_esperado)
    .limit(1).count()
):
    raise ValueError(
        "O Delta diverge do resumo preparado. "
        "Confira a execução da célula 36."
    )

duplicados = (
    df_regra09_publicacao_sql
    .groupBy("arquivo_origem")
    .count()
    .filter(F.col("count") > 1)
)

if duplicados.limit(1).count():
    raise ValueError("Há arquivos duplicados. Publicação cancelada.")

# Substitui as métricas antigas pelos dados atuais do Delta.
(
    df_regra09_publicacao_sql.write
    .format("sqlserver")
    .options(**opcoes_sql_regra09)
    .mode("overwrite")
    .save()
)

# Relê o SQL para validar a publicação.
df_sql_regra9 = (
    spark.read
    .format("sqlserver")
    .options(**opcoes_sql_regra09)
    .load()
)

df_regra09_sql_comparacao = df_sql_regra9.select(
    *[
        F.col(campo.name).cast(campo.dataType).alias(campo.name)
        for campo in df_regra09_publicacao_sql.schema.fields
    ]
)

faltando = (
    df_regra09_publicacao_sql
    .exceptAll(df_regra09_sql_comparacao)
    .limit(1).count()
)

sobrando = (
    df_regra09_sql_comparacao
    .exceptAll(df_regra09_publicacao_sql)
    .limit(1).count()
)

if faltando or sobrando:
    raise ValueError(
        "A gravação terminou, mas o SQL diverge do Delta."
    )

print("VALIDAÇÃO APROVADA: SQL corresponde ao Delta atualizado.")
print("Tabela:", opcoes_sql_regra09["dbtable"])
print("Lotes publicados:", len(linhas_regra09_sql))

display(
    df_sql_regra9.agg(
        F.sum("total_clientes").alias("total_registros"),
        F.sum("emails_provedores_reconhecidos").alias(
            "emails_reconhecidos"
        ),
        F.sum("qtd_emails_nao_reconhecidos").alias(
            "emails_nao_reconhecidos"
        )
    )
)

In [0]:
# ============================================================
# CONFERIR REGRA 09 — SQL X SILVER ATUAL
# ============================================================

from pyspark.sql import functions as F

df_sql_regra9 = (
    spark.read
    .format("sqlserver")
    .option("host", jdbc_hostname)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", "squad1.gold_clientes_regra09")
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("encrypt", "true")
    .load()
)

# Referência atual da Silver.
if dict(df_clientes_silver.dtypes).get("regra_09_falhou") != "boolean":
    raise ValueError("A flag regra_09_falhou precisa ser booleana.")

if df_clientes_silver.filter(
    F.col("regra_09_falhou").isNull()
).limit(1).count():
    raise ValueError("Há flags nulas na regra 09.")

referencia = df_clientes_silver.agg(
    F.count("*").alias("total"),
    F.sum(
        F.col("regra_09_falhou").cast("long")
    ).alias("falhas")
).first()

publicado = df_sql_regra9.agg(
    F.count("*").alias("lotes"),
    F.countDistinct("arquivo_origem").alias("arquivos_distintos"),
    F.sum("total_clientes").alias("total"),
    F.sum("qtd_emails_nao_reconhecidos").alias("falhas")
).first()

caminhos_invalidos = df_sql_regra9.filter(
    F.col("arquivo_origem").isNull()
    | ~F.col("arquivo_origem").rlike(
        r"^real-time-data/\d{4}/\d{2}/\d{2}/\d{6}/.+$"
    )
).count()

print("===== CONFERÊNCIA DA REGRA 09 =====")
print("Registros na Silver:", referencia["total"])
print("Falhas na Silver:", referencia["falhas"])
print("Registros contabilizados no SQL:", publicado["total"])
print("Falhas contabilizadas no SQL:", publicado["falhas"])
print("Lotes no SQL:", publicado["lotes"])
print("Arquivos distintos:", publicado["arquivos_distintos"])
print("Caminhos fora do padrão:", caminhos_invalidos)

totais_conferem = (
    publicado["total"] == referencia["total"]
    and publicado["falhas"] == referencia["falhas"]
)

if (
    totais_conferem
    and publicado["lotes"] == publicado["arquivos_distintos"]
    and caminhos_invalidos == 0
):
    print("Totais e caminhos conferem. Falta comparar por micro-lote.")
else:
    print("DIVERGÊNCIA ENCONTRADA: a publicação precisa de ajuste.")

print("Nenhum dado foi gravado.")

In [0]:
# CONSULTAR TABELAS NO AZURE SQL

from pyspark.sql import functions as F

df_tabelas = (
    spark.read
    .format("sqlserver")
    .option("host", jdbc_hostname)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option(
        "query",
        """
        SELECT TABLE_SCHEMA, TABLE_NAME
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_TYPE = 'BASE TABLE'
          AND (
              TABLE_NAME LIKE '%cliente%'
              OR TABLE_NAME LIKE '%regra%'
              OR TABLE_NAME LIKE '%qualidade%'
              OR TABLE_NAME LIKE '%dq%'
          )
        """
    )
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("encrypt", "true")
    .load()
)

print("===== TABELAS ENCONTRADAS NO AZURE SQL =====")

display(
    df_tabelas.orderBy(
        F.col("TABLE_SCHEMA"),
        F.col("TABLE_NAME")
    )
)

print("Total de tabelas:", df_tabelas.count())

In [0]:
from pyspark.sql import DataFrame

print("DATAFRAMES DISPONÍVEIS:")

for nome, valor in list(globals().items()):
    if isinstance(valor, DataFrame):
        print(nome)


In [0]:
print("TOTAL DE LINHAS:")
print(gold_dq_resumo_por_regra.count())

print("COLUNAS:")
print(gold_dq_resumo_por_regra.columns)

print("RESUMO DAS REGRAS:")
display(gold_dq_resumo_por_regra)

In [0]:
from pyspark.sql import functions as F

df_regras_clientes = (
    gold_dq_resumo_por_regra
    .filter(F.col("tabela") == "ecommerce_clientes")
)

print("TOTAL DE LINHAS DE CLIENTES:")
print(df_regras_clientes.count())

print("REGRAS DISTINTAS DE CLIENTES:")
display(
    df_regras_clientes
    .select("regra")
    .distinct()
    .orderBy("regra")
)

print("RESUMO COMPLETO DE CLIENTES:")
display(
    df_regras_clientes
    .orderBy("data_execucao", "regra")
)

In [0]:
# ============================================================
# SEVERIDADES CONFORME A PLANILHA OFICIAL — SQUAD 1
# Regras 01–05: técnicas | Regras 06–10: negócio
# ============================================================

colunas_validacao = [
    f"regra_{i:02d}_falhou"
    for i in range(1, 11)
]

tipos = dict(df_clientes_silver.dtypes)

invalidas = [
    regra for regra in colunas_validacao
    if tipos.get(regra) != "boolean"
]

if invalidas:
    raise ValueError(
        f"Flags ausentes ou não booleanas: {invalidas}"
    )

severidades_regras = {
    "regra_01_falhou": "Critica",
    "regra_02_falhou": "Critica",
    "regra_03_falhou": "Critica",
    "regra_04_falhou": "Critica",
    "regra_05_falhou": "Critica",
    "regra_06_falhou": "Aviso",
    "regra_07_falhou": "Aviso",
    "regra_08_falhou": "Aviso",
    "regra_09_falhou": "Aviso",
    "regra_10_falhou": "Aviso",
}

df_severidades_clientes = spark.createDataFrame(
    list(severidades_regras.items()),
    schema="regra string, severidade string"
)

df_severidades_clientes.orderBy("regra").show(
    10, truncate=False
)

print("Mapeamento: 5 regras técnicas e 5 de negócio.")
print("Nenhum dado foi gravado no Delta ou no SQL.")

In [0]:
# ============================================================
# RESUMO POR REGRA E DATA/HORA DO MICRO-LOTE
# Não grava dados
# ============================================================

from pyspark.sql import functions as F
from functools import reduce

regras = [f"regra_{i:02d}_falhou" for i in range(1, 11)]

# Conferir o mapeamento
if set(severidades_regras) != set(regras):
    raise ValueError("O mapeamento deve conter as dez regras.")

# Conferir flags nulas antes da contagem
tem_nulo = reduce(
    lambda a, b: a | b,
    [F.col(regra).isNull() for regra in regras]
)

if df_base_limpeza.filter(tem_nulo).limit(1).count():
    raise ValueError("Existem flags nulas na base.")

if df_base_limpeza.filter(
    F.col("_timestamp_lote").isNull()
).limit(1).count():
    raise ValueError("Existem micro-lotes sem data/hora.")

# Uma linha por regra e data/hora do lote,
# incluindo regras com zero falhas.
resumos = []

for regra in regras:
    resumo = (
        df_base_limpeza
        .groupBy("_timestamp_lote")
        .agg(
            F.sum(
                F.col(regra).cast("long")
            ).alias("qtd_falhas"),
            F.count("*").alias("qtd_total")
        )
        .withColumnRenamed("_timestamp_lote", "_data")
        .withColumn("tabela", F.lit("ecommerce_clientes"))
        .withColumn("regra", F.lit(regra))
        .withColumn(
            "severidade",
            F.lit(severidades_regras[regra])
        )
        .withColumn(
            "pct_falha",
            F.round(
                F.col("qtd_falhas") * 100.0
                / F.col("qtd_total"),
                2
            )
        )
        .select(
            "tabela",
            "regra",
            "_data",
            "qtd_falhas",
            "qtd_total",
            "pct_falha",
            "severidade"
        )
    )

    resumos.append(resumo)

gold_clientes_resumo_geral = reduce(
    lambda a, b: a.unionByName(b),
    resumos
)

# Conferir cobertura das dez regras
total_base = df_base_limpeza.count()

conferencia_regras = (
    gold_clientes_resumo_geral
    .groupBy("regra", "severidade")
    .agg(
        F.sum("qtd_falhas").alias("qtd_falhas"),
        F.sum("qtd_total").alias("qtd_total"),
        F.count("*").alias("qtd_periodos")
    )
    .withColumn(
        "pct_falha",
        F.round(
            F.col("qtd_falhas") * 100.0
            / F.col("qtd_total"),
            2
        )
    )
)

if conferencia_regras.filter(
    F.col("qtd_total") != total_base
).limit(1).count():
    raise ValueError("O total por regra difere da base.")

print("===== CONFERÊNCIA DAS DEZ REGRAS =====")
display(conferencia_regras.orderBy("regra"))

print("===== RESUMO POR MICRO-LOTE =====")
print("Linhas preparadas:", gold_clientes_resumo_geral.count())

display(
    gold_clientes_resumo_geral
    .orderBy("_data", "regra")
    .limit(30)
)

print("Resumo preparado. Nenhum dado foi gravado.")

In [0]:
# ============================================================
# PUBLICAR E VALIDAR O RESUMO DAS 10 REGRAS NO AZURE SQL
# ============================================================

from pyspark.sql import functions as F
from functools import reduce

tabela_sql_regras = "squad1.gold_clientes_resumo_geral"

colunas_resumo_regras = [
    "tabela",
    "regra",
    "_data",
    "qtd_falhas",
    "qtd_total",
    "pct_falha",
    "severidade",
]

df_publicacao_regras = (
    gold_clientes_resumo_geral
    .select(*colunas_resumo_regras)
    .cache()
)

# 1. Validar antes de publicar
total_linhas = df_publicacao_regras.count()

if total_linhas == 0:
    raise ValueError("O resumo por regra está vazio.")

tem_nulo = reduce(
    lambda a, b: a | b,
    [F.col(c).isNull() for c in colunas_resumo_regras]
)

if df_publicacao_regras.filter(tem_nulo).limit(1).count():
    raise ValueError("Existem valores nulos no resumo.")

duplicados = (
    df_publicacao_regras
    .groupBy("tabela", "regra", "_data")
    .count()
    .filter(F.col("count") > 1)
)

if duplicados.limit(1).count():
    raise ValueError("Existem regras duplicadas no mesmo período.")

if df_publicacao_regras.filter(
    (F.col("qtd_total") <= 0)
    | (F.col("qtd_falhas") < 0)
    | (F.col("qtd_falhas") > F.col("qtd_total"))
).limit(1).count():
    raise ValueError("Existem contagens inválidas.")

cobertura = (
    df_publicacao_regras
    .groupBy("tabela", "_data")
    .agg(F.countDistinct("regra").alias("qtd_regras"))
)

if cobertura.filter(
    F.col("qtd_regras") != 10
).limit(1).count():
    raise ValueError("Há períodos sem as dez regras.")

# 2. Configurar a conexão
opcoes_sql_regras = {
    "host": jdbc_hostname,
    "port": "1433",
    "database": jdbc_database,
    "dbtable": tabela_sql_regras,
    "user": jdbc_username,
    "password": jdbc_password,
    "encrypt": "true",
}

# 3. Substituir o resumo antigo pelo resumo completo atualizado
(
    df_publicacao_regras.write
    .format("sqlserver")
    .options(**opcoes_sql_regras)
    .mode("overwrite")
    .save()
)

# 4. Reler o SQL e comparar todo o conteúdo
df_sql_resumo_regras = (
    spark.read
    .format("sqlserver")
    .options(**opcoes_sql_regras)
    .load()
    .select(*colunas_resumo_regras)
)

tipos_origem = dict(df_publicacao_regras.dtypes)

df_sql_regras_comparacao = df_sql_resumo_regras.select(
    *[
        F.col(c).cast(tipos_origem[c]).alias(c)
        for c in colunas_resumo_regras
    ]
)

faltando = (
    df_publicacao_regras
    .exceptAll(df_sql_regras_comparacao)
    .limit(1)
    .count()
)

sobrando = (
    df_sql_regras_comparacao
    .exceptAll(df_publicacao_regras)
    .limit(1)
    .count()
)

if faltando or sobrando:
    raise ValueError("O conteúdo publicado difere do resumo preparado.")

print("VALIDAÇÃO APROVADA: SQL corresponde ao resumo preparado.")
print("Tabela:", tabela_sql_regras)
print("Linhas publicadas:", total_linhas)

display(
    df_sql_resumo_regras
    .groupBy("regra", "severidade")
    .agg(
        F.sum("qtd_falhas").alias("qtd_falhas"),
        F.sum("qtd_total").alias("qtd_total"),
        F.count("*").alias("qtd_periodos")
    )
    .withColumn(
        "pct_falha",
        F.round(
            F.col("qtd_falhas") * 100.0 / F.col("qtd_total"),
            2
        )
    )
    .orderBy("regra")
)

df_publicacao_regras.unpersist()